## LLM Problem Definition

In [1]:
import pandas as pd

customer_features = pd.read_csv("../data/customer_features.csv")
product_features = pd.read_csv("../data/product_features.csv")
order_analytics = pd.read_csv("../data/order_analytics.csv")

print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)

Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)


In [2]:
llm_use_cases = {
    "Business Q&A": "Answer questions about sales, customers, products, and orders.",
    "Customer Insights": "Summarize customer behavior and identify important customer segments.",
    "Product Insights": "Explain product performance and identify important products.",
    "Trend Analysis": "Summarize important business trends from analytical results.",
    "Recommendation Support": "Turn analytical findings into business recommendations.",
    "Report Generation": "Generate concise natural-language business summaries."
}

for use_case, description in llm_use_cases.items():
    print(f"{use_case}: {description}")

Business Q&A: Answer questions about sales, customers, products, and orders.
Customer Insights: Summarize customer behavior and identify important customer segments.
Product Insights: Explain product performance and identify important products.
Trend Analysis: Summarize important business trends from analytical results.
Recommendation Support: Turn analytical findings into business recommendations.
Report Generation: Generate concise natural-language business summaries.


In [3]:
initial_llm_task = {
    "input": "A structured business question plus relevant CommerceIQ data",
    "processing": "LLM interprets the question and analytical context",
    "output": "Clear, concise business-oriented answer"
}

for key, value in initial_llm_task.items():
    print(f"{key}: {value}")

input: A structured business question plus relevant CommerceIQ data
processing: LLM interprets the question and analytical context
output: Clear, concise business-oriented answer


In [4]:
example_questions = [
    "Which customer segments generate the most revenue?",
    "Which products are the strongest revenue contributors?",
    "How dependent is the business on the UK market?",
    "Which customers may need retention attention?",
    "What are the most important business trends?"
]

for i, question in enumerate(example_questions, start=1):
    print(f"{i}. {question}")

1. Which customer segments generate the most revenue?
2. Which products are the strongest revenue contributors?
3. How dependent is the business on the UK market?
4. Which customers may need retention attention?
5. What are the most important business trends?


## Prompting Basics

In [5]:
business_question = "Which customer segments generate the most revenue?"

basic_prompt = f"""
You are a business analytics assistant for an e-commerce company.

Answer the following question clearly and concisely:

{business_question}
"""

print(basic_prompt)


You are a business analytics assistant for an e-commerce company.

Answer the following question clearly and concisely:

Which customer segments generate the most revenue?



In [6]:
business_context = """
CommerceIQ analyzes e-commerce transactions, customers, products,
orders, revenue, and customer segments.

The assistant should use only the information provided in the context
and should avoid inventing numerical results.
"""

context_prompt = f"""
You are CommerceIQ, an e-commerce business analytics assistant.

Business context:
{business_context}

Question:
{business_question}

Provide a concise, business-focused answer.
"""

print(context_prompt)


You are CommerceIQ, an e-commerce business analytics assistant.

Business context:

CommerceIQ analyzes e-commerce transactions, customers, products,
orders, revenue, and customer segments.

The assistant should use only the information provided in the context
and should avoid inventing numerical results.


Question:
Which customer segments generate the most revenue?

Provide a concise, business-focused answer.



In [7]:
structured_prompt = f"""
ROLE:
You are CommerceIQ, an e-commerce analytics assistant.

TASK:
Answer the user's business question using the provided data.

RULES:
- Use only the provided information.
- Do not invent numbers or facts.
- Clearly distinguish facts from interpretations.
- Keep the answer concise.
- Mention important numbers when available.

BUSINESS QUESTION:
{business_question}

OUTPUT:
Provide:
1. Direct answer
2. Supporting evidence
3. Business implication
"""

print(structured_prompt)


ROLE:
You are CommerceIQ, an e-commerce analytics assistant.

TASK:
Answer the user's business question using the provided data.

RULES:
- Use only the provided information.
- Do not invent numbers or facts.
- Clearly distinguish facts from interpretations.
- Keep the answer concise.
- Mention important numbers when available.

BUSINESS QUESTION:
Which customer segments generate the most revenue?

OUTPUT:
Provide:
1. Direct answer
2. Supporting evidence
3. Business implication



In [8]:
prompts = {
    "Basic": basic_prompt,
    "Context-Aware": context_prompt,
    "Structured": structured_prompt
}

for name, prompt in prompts.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(prompt)

Basic

You are a business analytics assistant for an e-commerce company.

Answer the following question clearly and concisely:

Which customer segments generate the most revenue?

Context-Aware

You are CommerceIQ, an e-commerce business analytics assistant.

Business context:

CommerceIQ analyzes e-commerce transactions, customers, products,
orders, revenue, and customer segments.

The assistant should use only the information provided in the context
and should avoid inventing numerical results.


Question:
Which customer segments generate the most revenue?

Provide a concise, business-focused answer.

Structured

ROLE:
You are CommerceIQ, an e-commerce analytics assistant.

TASK:
Answer the user's business question using the provided data.

RULES:
- Use only the provided information.
- Do not invent numbers or facts.
- Clearly distinguish facts from interpretations.
- Keep the answer concise.
- Mention important numbers when available.

BUSINESS QUESTION:
Which customer segments gene

In [9]:
business_question_2 = "How dependent is the business on the UK market?"

uk_prompt = f"""
ROLE:
You are CommerceIQ, an e-commerce analytics assistant.

TASK:
Analyze the provided business data and answer the question.

RULES:
- Do not invent data.
- Use numerical evidence where available.
- Explain the business implication.
- Keep the response concise.

QUESTION:
{business_question_2}

OUTPUT FORMAT:
Answer:
Evidence:
Business implication:
"""

print(uk_prompt)


ROLE:
You are CommerceIQ, an e-commerce analytics assistant.

TASK:
Analyze the provided business data and answer the question.

RULES:
- Do not invent data.
- Use numerical evidence where available.
- Explain the business implication.
- Keep the response concise.

QUESTION:
How dependent is the business on the UK market?

OUTPUT FORMAT:
Answer:
Evidence:
Business implication:



## LLM Model Setup

In [10]:
%pip install -q groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from groq import Groq
from dotenv import load_dotenv

print("Imports successful")

Imports successful


In [2]:
load_dotenv("../.env")

api_key = os.getenv("GROQ_API_KEY")

print("API key loaded:", api_key is not None)
print("API key length:", len(api_key) if api_key else 0)

API key loaded: True
API key length: 56


In [3]:
client = Groq(api_key=api_key)

print("Groq client created successfully")

Groq client created successfully


In [5]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain e-commerce customer retention in one sentence."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

E‑commerce customer retention is the set of strategies and practices designed to keep shoppers returning for repeat purchases, thereby boosting their lifetime value and sustaining long‑term revenue.


In [6]:
print("Model:", response.model)
print("Finish reason:", response.choices[0].finish_reason)

if response.usage:
    print("Prompt tokens:", response.usage.prompt_tokens)
    print("Completion tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)

Model: openai/gpt-oss-20b
Finish reason: stop
Prompt tokens: 80
Completion tokens: 95
Total tokens: 175


## Structured LLM Outputs

In [8]:
business_question = "How dependent is the business on the UK market?"

structured_prompt = f"""
You are CommerceIQ, an e-commerce analytics assistant.

Answer the following question.

Question:
{business_question}

Return your response using exactly these sections:

Answer:
Evidence:
Business Implication:

Rules:
- Use only information provided in the question or context.
- Do not invent numbers.
- Keep the response concise.
"""
print(structured_prompt)


You are CommerceIQ, an e-commerce analytics assistant.

Answer the following question.

Question:
How dependent is the business on the UK market?

Return your response using exactly these sections:

Answer:
Evidence:
Business Implication:

Rules:
- Use only information provided in the question or context.
- Do not invent numbers.
- Keep the response concise.



In [9]:
uk_revenue = 8941988.244
total_revenue = 10539552.834

uk_share = (uk_revenue / total_revenue) * 100
international_share = 100 - uk_share

data_context = f"""
Total sales revenue: £{total_revenue:,.2f}
UK revenue: £{uk_revenue:,.2f}
UK revenue share: {uk_share:.2f}%
International revenue share: {international_share:.2f}%
"""

print(data_context)


Total sales revenue: £10,539,552.83
UK revenue: £8,941,988.24
UK revenue share: 84.84%
International revenue share: 15.16%



In [10]:
final_prompt = f"""
You are CommerceIQ, an e-commerce analytics assistant.

Business data:
{data_context}

Question:
{business_question}

Return exactly:

Answer:
Evidence:
Business Implication:

Rules:
- Use only the provided business data.
- Do not invent numbers.
- Keep the answer concise.
- Explain what the numbers mean for the business.
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": final_prompt
        }
    ],
    temperature=0
)

structured_response = response.choices[0].message.content

print(structured_response)

Answer:  
The business is highly dependent on the UK market, with 84.84 % of total revenue coming from the UK.

Evidence:  
- Total sales revenue: £10,539,552.83  
- UK revenue: £8,941,988.24  
- UK revenue share: 84.84 % (calculated as £8,941,988.24 ÷ £10,539,552.83)

Business Implication:  
A reliance on a single market exposes the company to regional economic fluctuations, regulatory changes, and competitive pressures. Diversifying the international revenue base (currently 15.16 %) could reduce risk and support more stable growth.


In [11]:
def parse_sections(text):
    sections = {
        "Answer": "",
        "Evidence": "",
        "Business Implication": ""
    }

    current_section = None

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("Answer:"):
            current_section = "Answer"
            sections[current_section] = line.replace(
                "Answer:", "", 1
            ).strip()

        elif line.startswith("Evidence:"):
            current_section = "Evidence"
            sections[current_section] = line.replace(
                "Evidence:", "", 1
            ).strip()

        elif line.startswith("Business Implication:"):
            current_section = "Business Implication"
            sections[current_section] = line.replace(
                "Business Implication:", "", 1
            ).strip()

        elif current_section and line:
            sections[current_section] += " " + line

    return sections


parsed_response = parse_sections(structured_response)

for section, content in parsed_response.items():
    print(f"\n{section}:")
    print(content)


Answer:
 The business is highly dependent on the UK market, with 84.84 % of total revenue coming from the UK.

Evidence:
 - Total sales revenue: £10,539,552.83 - UK revenue: £8,941,988.24 - UK revenue share: 84.84 % (calculated as £8,941,988.24 ÷ £10,539,552.83)

Business Implication:
 A reliance on a single market exposes the company to regional economic fluctuations, regulatory changes, and competitive pressures. Diversifying the international revenue base (currently 15.16 %) could reduce risk and support more stable growth.


In [13]:
import pandas as pd
structured_output_df = pd.DataFrame(
    [parsed_response]
)

display(structured_output_df)

,Answer,Evidence,Business Implication
0,The business is highly dependent on the UK ma...,"- Total sales revenue: £10,539,552.83 - UK re...",A reliance on a single market exposes the com...


## Business Question → LLM Response

In [14]:
def ask_commerceiq(question, context):
    prompt = f"""
You are CommerceIQ, an e-commerce analytics assistant.

Business data:
{context}

Business question:
{question}

Return exactly these sections:

Answer:
Evidence:
Business Implication:

Rules:
- Use only the provided business data.
- Do not invent numbers or facts.
- Clearly distinguish facts from interpretation.
- Keep the response concise.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [15]:
commerceiq_context = f"""
Overall sales revenue: £10,539,552.83
Net product revenue after cancellations: £10,027,639.15
Total orders: 19,865
Unique customers: 4,335
Unique products: 3,918

UK revenue: £8,941,988.24
UK revenue share: 84.84%
International revenue share: 15.16%

Top customer segment by revenue:
Champions — £5,765,056.43

Top 10 customer revenue share: 17.37%
Top 10 product revenue share: 10.80%

Active customers:
1,647 customers
Revenue: £6,224,125.47
Revenue share: 70.46%

At Risk customers:
586 customers
Revenue: £467,247.21
Revenue share: 5.29%
"""

print(commerceiq_context)


Overall sales revenue: £10,539,552.83
Net product revenue after cancellations: £10,027,639.15
Total orders: 19,865
Unique customers: 4,335
Unique products: 3,918

UK revenue: £8,941,988.24
UK revenue share: 84.84%
International revenue share: 15.16%

Top customer segment by revenue:
Champions — £5,765,056.43

Top 10 customer revenue share: 17.37%
Top 10 product revenue share: 10.80%

Active customers:
1,647 customers
Revenue: £6,224,125.47
Revenue share: 70.46%

At Risk customers:
586 customers
Revenue: £467,247.21
Revenue share: 5.29%



In [16]:
question = "Which customer segment generates the most revenue?"

answer = ask_commerceiq(
    question,
    commerceiq_context
)

print(answer)

Answer:  
Active customers generate the most revenue.

Evidence:  
- Active customers: £6,224,125.47  
- Champions: £5,765,056.43  
- At Risk customers: £467,247.21  

Business Implication:  
The highest revenue comes from the active customer base, indicating that maintaining and expanding this segment should be a priority for sustaining growth and maximizing profitability.


In [17]:
questions = [
    "Which customer segment generates the most revenue?",
    "How dependent is the business on the UK market?",
    "Which customer group may need retention attention?",
    "How concentrated is revenue among the top customers and products?"
]

for question in questions:
    print("=" * 80)
    print("QUESTION:", question)
    
    answer = ask_commerceiq(
        question,
        commerceiq_context
    )
    
    print(answer)
    print()

QUESTION: Which customer segment generates the most revenue?
Answer:  
Champions

Evidence:  
- Top customer segment by revenue: Champions — £5,765,056.43

Business Implication:  
Champions represent the highest‑value customer group, so targeted retention, upsell, and loyalty initiatives should focus on this segment to sustain and grow revenue.

QUESTION: How dependent is the business on the UK market?
**Answer:**  
The business derives the vast majority of its revenue from the UK, with 84.84 % of total sales coming from that market.

**Evidence:**  
- Total revenue: £10,539,552.83  
- UK revenue: £8,941,988.24 (84.84 % of total)  
- International revenue share: 15.16 %

**Business Implication:**  
The company is highly dependent on the UK market, exposing it to significant risk if UK demand weakens. Diversifying revenue streams and strengthening international channels should be a strategic priority to reduce this concentration risk.

QUESTION: Which customer group may need retention a

In [18]:
print("Number of questions:", len(questions))
print("All questions processed successfully:", len(questions) == 4)

Number of questions: 4
All questions processed successfully: True


## LLM + CommerceIQ Data

In [19]:
import pandas as pd

customer_features = pd.read_csv("../data/customer_features.csv")
product_features = pd.read_csv("../data/product_features.csv")
order_analytics = pd.read_csv("../data/order_analytics.csv")

print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)

Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)


In [20]:
customer_context = f"""
CUSTOMER ANALYTICS

Total identifiable customers: {len(customer_features):,}

Total customer revenue:
£{customer_features["Total_Revenue"].sum():,.2f}

Average customer revenue:
£{customer_features["Total_Revenue"].mean():,.2f}

Median customer revenue:
£{customer_features["Total_Revenue"].median():,.2f}

Average orders per customer:
{customer_features["Total_Orders"].mean():.2f}

Top customer by revenue:
Customer {customer_features.loc[
    customer_features["Total_Revenue"].idxmax(), "Customer ID"
]}
with £{customer_features["Total_Revenue"].max():,.2f}
"""

print(customer_context)


CUSTOMER ANALYTICS

Total identifiable customers: 4,335

Total customer revenue:
£8,833,806.96

Average customer revenue:
£2,037.79

Median customer revenue:
£668.56

Average orders per customer:
4.26

Top customer by revenue:
Customer 14646.0
with £280,206.02



In [21]:
segment_summary = (
    customer_features
    .groupby("Revenue_Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
        Average_Revenue=("Total_Revenue", "mean")
    )
    .sort_values("Total_Revenue", ascending=False)
)

print(segment_summary)

segment_context = "\nCUSTOMER REVENUE SEGMENTS\n"

for segment, row in segment_summary.iterrows():
    segment_context += (
        f"- {segment}: "
        f"{int(row['Customers']):,} customers, "
        f"£{row['Total_Revenue']:,.2f} total revenue, "
        f"£{row['Average_Revenue']:,.2f} average revenue\n"
    )

print(segment_context)

                 Customers  Total_Revenue  Average_Revenue
Revenue_Segment                                           
Very High             1084    6988949.301      6447.370204
High                  1083    1152263.231      1063.954969
Medium                1084     500073.492       461.322410
Low                   1084     192520.940       177.602343

CUSTOMER REVENUE SEGMENTS
- Very High: 1,084 customers, £6,988,949.30 total revenue, £6,447.37 average revenue
- High: 1,083 customers, £1,152,263.23 total revenue, £1,063.95 average revenue
- Medium: 1,084 customers, £500,073.49 total revenue, £461.32 average revenue
- Low: 1,084 customers, £192,520.94 total revenue, £177.60 average revenue



In [22]:
top_products = (
    product_features
    .sort_values("Total_Revenue", ascending=False)
    .head(10)
)

product_context = "\nTOP 10 PRODUCTS BY REVENUE\n"

for _, row in top_products.iterrows():
    product_context += (
        f"- {row['Product_Name']}: "
        f"£{row['Total_Revenue']:,.2f} revenue, "
        f"{int(row['Total_Units_Sold']):,} units sold\n"
    )

print(product_context)


TOP 10 PRODUCTS BY REVENUE
- DOTCOM POSTAGE: £206,248.77 revenue, 706 units sold
- REGENCY CAKESTAND 3 TIER: £174,156.54 revenue, 13,851 units sold
- PAPER CRAFT , LITTLE BIRDIE: £168,469.60 revenue, 80,995 units sold
- WHITE HANGING HEART T-LIGHT HOLDER: £104,462.75 revenue, 37,641 units sold
- PARTY BUNTING: £99,445.23 revenue, 18,283 units sold
- JUMBO BAG RED RETROSPOT: £94,159.81 revenue, 48,371 units sold
- MEDIUM CERAMIC TOP STORAGE JAR: £81,700.92 revenue, 78,033 units sold
- POSTAGE: £78,119.88 revenue, 3,151 units sold
- RABBIT NIGHT LIGHT: £66,870.03 revenue, 30,739 units sold
- PAPER CHAIN KIT 50'S CHRISTMAS : £64,875.59 revenue, 19,329 units sold



In [25]:
order_context = f"""
ORDER ANALYTICS

Total orders: {len(order_analytics):,}

Average order value:
£{order_analytics["Order_Revenue"].mean():,.2f}

Median order value:
£{order_analytics["Order_Revenue"].median():,.2f}

Average units per order:
{order_analytics["Units"].mean():,.2f}

Median units per order:
{order_analytics["Units"].median():,.2f}

Average unique products per order:
{order_analytics["Unique_Products"].mean():,.2f}

Median unique products per order:
{order_analytics["Unique_Products"].median():,.2f}
"""

print(order_context)


ORDER ANALYTICS

Total orders: 19,865

Average order value:
£530.56

Median order value:
£303.20

Average units per order:
280.16

Median units per order:
152.00

Average unique products per order:
26.14

Median unique products per order:
15.00



In [26]:
full_commerceiq_context = (
    customer_context
    + segment_context
    + product_context
    + order_context
)

print(full_commerceiq_context)


CUSTOMER ANALYTICS

Total identifiable customers: 4,335

Total customer revenue:
£8,833,806.96

Average customer revenue:
£2,037.79

Median customer revenue:
£668.56

Average orders per customer:
4.26

Top customer by revenue:
Customer 14646.0
with £280,206.02

CUSTOMER REVENUE SEGMENTS
- Very High: 1,084 customers, £6,988,949.30 total revenue, £6,447.37 average revenue
- High: 1,083 customers, £1,152,263.23 total revenue, £1,063.95 average revenue
- Medium: 1,084 customers, £500,073.49 total revenue, £461.32 average revenue
- Low: 1,084 customers, £192,520.94 total revenue, £177.60 average revenue

TOP 10 PRODUCTS BY REVENUE
- DOTCOM POSTAGE: £206,248.77 revenue, 706 units sold
- REGENCY CAKESTAND 3 TIER: £174,156.54 revenue, 13,851 units sold
- PAPER CRAFT , LITTLE BIRDIE: £168,469.60 revenue, 80,995 units sold
- WHITE HANGING HEART T-LIGHT HOLDER: £104,462.75 revenue, 37,641 units sold
- PARTY BUNTING: £99,445.23 revenue, 18,283 units sold
- JUMBO BAG RED RETROSPOT: £94,159.81 reve

In [27]:
question = "Which customer revenue segment generates the most revenue?"

answer = ask_commerceiq(
    question,
    full_commerceiq_context
)

print(answer)

Answer:  
The **Very High** customer revenue segment generates the most revenue.

Evidence:  
- Very High: £6,988,949.30  
- High: £1,152,263.23  
- Medium: £500,073.49  
- Low: £192,520.94  

Business Implication:  
Prioritise retention, upsell, and targeted marketing for Very High customers to sustain and grow the largest revenue source.


In [28]:
question_2 = "Which products are the strongest revenue contributors?"

answer_2 = ask_commerceiq(
    question_2,
    full_commerceiq_context
)

print(answer_2)

**Answer:**  
The strongest revenue contributors are the top‑10 products listed below, each generating the highest revenue for the business.

**Evidence:**  
| Rank | Product | Revenue (£) | Units Sold |
|------|---------|-------------|------------|
| 1 | DOTCOM POSTAGE | 206,248.77 | 706 |
| 2 | REGENCY CAKESTAND 3 TIER | 174,156.54 | 13,851 |
| 3 | PAPER CRAFT, LITTLE BIRDIE | 168,469.60 | 80,995 |
| 4 | WHITE HANGING HEART T‑LIGHT HOLDER | 104,462.75 | 37,641 |
| 5 | PARTY BUNTING | 99,445.23 | 18,283 |
| 6 | JUMBO BAG RED RETROSPOT | 94,159.81 | 48,371 |
| 7 | MEDIUM CERAMIC TOP STORAGE JAR | 81,700.92 | 78,033 |
| 8 | POSTAGE | 78,119.88 | 3,151 |
| 9 | RABBIT NIGHT LIGHT | 66,870.03 | 30,739 |
| 10 | PAPER CHAIN KIT 50'S CHRISTMAS | 64,875.59 | 19,329 |

**Business Implication:**  
- Prioritize inventory, supply chain, and marketing focus on these top‑10 items to sustain and grow revenue.  
- Consider bundling or cross‑selling these high‑volume products with complementary items t

## Basic Guardrails

In [29]:
def ask_commerceiq_safe(question, context):
    prompt = f"""
You are CommerceIQ, an e-commerce analytics assistant.

BUSINESS DATA:
{context}

BUSINESS QUESTION:
{question}

Return exactly these sections:

Answer:
Evidence:
Business Implication:
Caveat:

Rules:
1. Use ONLY the provided business data.
2. Never invent numbers, products, customers, or facts.
3. Every numerical claim must come from the provided data.
4. Clearly distinguish facts from business interpretation.
5. If the question is ambiguous, state the ambiguity.
6. Do not treat postage, fees, charges, samples, or administrative items as normal merchandise.
7. If the data is insufficient to answer confidently, say so.
8. Keep the answer concise and business-oriented.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [30]:
safe_context = f"""
CUSTOMER REVENUE SEGMENTS
{segment_context}

TOP PRODUCTS BY REVENUE
{product_context}

ORDER ANALYTICS
{order_context}

IMPORTANT DATA INTERPRETATION:
- Revenue_Segment represents customer revenue quartiles.
- Very High means customers in the highest revenue quartile.
- Postage, DOTCOM POSTAGE, bank charges, samples, and similar entries are not normal merchandise products.
"""

print(safe_context)


CUSTOMER REVENUE SEGMENTS

CUSTOMER REVENUE SEGMENTS
- Very High: 1,084 customers, £6,988,949.30 total revenue, £6,447.37 average revenue
- High: 1,083 customers, £1,152,263.23 total revenue, £1,063.95 average revenue
- Medium: 1,084 customers, £500,073.49 total revenue, £461.32 average revenue
- Low: 1,084 customers, £192,520.94 total revenue, £177.60 average revenue


TOP PRODUCTS BY REVENUE

TOP 10 PRODUCTS BY REVENUE
- DOTCOM POSTAGE: £206,248.77 revenue, 706 units sold
- REGENCY CAKESTAND 3 TIER: £174,156.54 revenue, 13,851 units sold
- PAPER CRAFT , LITTLE BIRDIE: £168,469.60 revenue, 80,995 units sold
- WHITE HANGING HEART T-LIGHT HOLDER: £104,462.75 revenue, 37,641 units sold
- PARTY BUNTING: £99,445.23 revenue, 18,283 units sold
- JUMBO BAG RED RETROSPOT: £94,159.81 revenue, 48,371 units sold
- MEDIUM CERAMIC TOP STORAGE JAR: £81,700.92 revenue, 78,033 units sold
- POSTAGE: £78,119.88 revenue, 3,151 units sold
- RABBIT NIGHT LIGHT: £66,870.03 revenue, 30,739 units sold
- PAPE

In [31]:
question = "Which customer segment generates the most revenue?"

response = ask_commerceiq_safe(
    question,
    safe_context
)

print(response)

Answer:  
The **Very High** customer segment generates the most revenue.

Evidence:  
- Very High: £6,988,949.30 total revenue  
- High: £1,152,263.23 total revenue  
- Medium: £500,073.49 total revenue  
- Low: £192,520.94 total revenue  

Business Implication:  
Focusing marketing, retention, and upsell efforts on Very High customers can maximize revenue growth, as they contribute the largest share of sales.

Caveat:  
This conclusion is based solely on the provided revenue totals; it does not account for other factors such as customer acquisition cost or lifetime value.


In [32]:
question = "Which products are the strongest merchandise revenue contributors?"

response = ask_commerceiq_safe(
    question,
    safe_context
)

print(response)

**Answer:**  
The strongest merchandise revenue contributors are the following products (excluding postage, DOTCOM POSTAGE, bank charges, samples, and other non‑merchandise items):

1. **REGENCY CAKESTAND 3 TIER** – £174,156.54  
2. **PAPER CRAFT, LITTLE BIRDIE** – £168,469.60  
3. **WHITE HANGING HEART T‑LIGHT HOLDER** – £104,462.75  
4. **PARTY BUNTING** – £99,445.23  
5. **JUMBO BAG RED RETROSPOT** – £94,159.81  
6. **MEDIUM CERAMIC TOP STORAGE JAR** – £81,700.92  
7. **RABBIT NIGHT LIGHT** – £66,870.03  
8. **PAPER CHAIN KIT 50'S CHRISTMAS** – £64,875.59  

**Evidence:**  
These figures come directly from the “Top 10 Products by Revenue” list provided, with postage‑related items omitted as per the rules.

**Business Implication:**  
These products generate the bulk of merchandise revenue and should be prioritized for inventory management, promotional campaigns, and supply‑chain optimization. Focusing on these high‑revenue items can maximize profitability and improve customer satisf

In [33]:
question = "Which customer generated £1 million in revenue?"

response = ask_commerceiq_safe(
    question,
    safe_context
)

print(response)

Answer:  
The provided data does not identify any single customer that generated exactly £1 million in revenue.

Evidence:  
- Customer revenue segments give only averages:  
  - Very High: £6,447.37 per customer  
  - High: £1,063.95 per customer  
  - Medium: £461.32 per customer  
  - Low: £177.60 per customer  
- No individual customer revenue figures are listed.

Business Implication:  
Without individual customer revenue data, we cannot pinpoint a £1 million‑earning customer or assess its impact on overall performance.

Caveat:  
The absence of granular customer revenue details means the question cannot be answered definitively from the supplied data.


## Conclusion

CommerceIQ now connects structured e-commerce analytics with an LLM for business question answering. The system supports structured responses, data-grounded reasoning, business interpretation, ambiguity handling, and basic hallucination guardrails.

In [35]:
print("VALIDATION")
print("-" * 40)

print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)

print("\nLLM model:", "openai/gpt-oss-20b")
print("Temperature:", 0)

print("\nCapabilities:")
print("✓ Business question answering")
print("✓ Structured LLM outputs")
print("✓ Data-grounded responses")
print("✓ Customer revenue analysis")
print("✓ Product revenue analysis")
print("✓ Business implications")
print("✓ Ambiguity handling")
print("✓ Basic hallucination guardrails")
print("✓ Non-merchandise product filtering")

VALIDATION
----------------------------------------
Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)

LLM model: openai/gpt-oss-20b
Temperature: 0

Capabilities:
✓ Business question answering
✓ Structured LLM outputs
✓ Data-grounded responses
✓ Customer revenue analysis
✓ Product revenue analysis
✓ Business implications
✓ Ambiguity handling
✓ Basic hallucination guardrails
✓ Non-merchandise product filtering


In [36]:
stage_8_summary = {
    "customer_features_rows": len(customer_features),
    "product_features_rows": len(product_features),
    "order_analytics_rows": len(order_analytics),
    "llm_model": "openai/gpt-oss-20b",
    "temperature": 0,
    "structured_outputs": True,
    "data_grounding": True,
    "guardrails": True,
    "business_qa": True
}

stage_8_summary

{'customer_features_rows': 4335,
 'product_features_rows': 3918,
 'order_analytics_rows': 19865,
 'llm_model': 'openai/gpt-oss-20b',
 'temperature': 0,
 'structured_outputs': True,
 'data_grounding': True,
 'guardrails': True,
 'business_qa': True}